# Wrangling Data Tabel Master
Tabel: hospitals, sensors, sites

## 0. Setup


In [1]:
   import pandas as pd
   pd.set_option("display.max_columns", None)
   print(pd.__version__)

3.1.1


## 2. Cleaning sites
### 2.1 site_type disederhanakan
Masalah: 19 nilai unik untuk 27 stasiun. Perbaikan: dikelompokkan jadi 6 kelompok.

In [20]:
def kelompok(t):
    if "Clean" in t:      return "Latar Bersih"
    if "Peatland" in t:   return "Gambut"
    if "Industrial" in t: return "Industri"
    if "Traffic" in t:    return "Lalu Lintas"
    if "Coastal" in t:    return "Pesisir"
    return "Perkotaan"

sites_clean = sites.copy()
sites_clean["site_group"] = sites_clean["site_type"].apply(kelompok)
sites_clean["site_group"].value_counts()

site_group
Gambut          8
Perkotaan       6
Industri        4
Latar Bersih    4
Pesisir         3
Lalu Lintas     2
Name: count, dtype: int64

### 2.2 Kolom pulau
Masalah: region_group tidak seragam dengan provinsi. Perbaikan: tambah kolom pulau dari provinsi.

In [21]:
pulau = {
    "DKI Jakarta": "Jawa", "Jawa Barat": "Jawa", "D.I. Yogyakarta": "Jawa", "Jawa Timur": "Jawa",
    "Kalimantan Tengah": "Kalimantan", "Kalimantan Barat": "Kalimantan",
    "Kalimantan Timur": "Kalimantan", "Kalimantan Selatan": "Kalimantan",
    "Aceh": "Sumatera", "Sumatera Barat": "Sumatera", "Riau": "Sumatera", "Jambi": "Sumatera",
    "Kepulauan Riau": "Sumatera", "Bengkulu": "Sumatera", "Sumatera Selatan": "Sumatera",
    "Lampung": "Sumatera", "Sumatera Utara": "Sumatera",
    "Sulawesi Selatan": "Sulawesi", "Sulawesi Tengah": "Sulawesi",
    "Papua Barat Daya": "Papua", "Papua Selatan": "Papua",
}
sites_clean["pulau"] = sites_clean["provinsi"].map(pulau)
sites_clean["pulau"].isna().sum()    # harus 0

np.int64(0)

### 2.3 Satuan
Masalah: 3 stasiun (Pesawaran, Medan, Banjarbaru) bersatuan ppm. Perbaikan: diseragamkan ke ug/m3, stasiun aslinya ditandai.

In [22]:
unit_cols = [c for c in sites_clean.columns if c.endswith("_unit")]
sites_clean["satuan_asli_ppm"] = sites_clean["pm25_unit"].eq("ppm")
sites_clean[unit_cols] = "ug/m3"
sites_clean["satuan_asli_ppm"].sum()   # 3

np.int64(3)

## 3. Cleaning sensors
Masalah: tanggal berupa teks, region_group tidak seragam, status bermasalah belum ditandai.

In [25]:
sens_clean = sens.copy()
sens_clean["tanggal_instalasi"] = pd.to_datetime(sens_clean["tanggal_instalasi"])
sens_clean = (sens_clean.drop(columns="region_group")
              .merge(sites_clean[["location_id", "provinsi", "pulau"]],
                     on="location_id", how="left", validate="one_to_one"))
sens_clean["status_bermasalah"] = sens_clean["status_operasional"].ne("Active")
sens_clean["status_bermasalah"].sum()   # 4

np.int64(4)

## 4. Cleaning hospitals
### 4.1 Kelas dan kapasitas
Masalah: kelas_faskes mencampur kelas dan jenis; kapasitas bertipe desimal.

In [26]:
hosp_clean = hosp.copy()
hosp_clean["kelas"]   = hosp_clean["kelas_faskes"].str.extract(r"Tipe ([ABC])")
hosp_clean["rs_paru"] = hosp_clean["kelas_faskes"].str.contains("Paru")
hosp_clean["kapasitas_tempat_tidur"] = hosp_clean["kapasitas_tempat_tidur"].astype("Int64")
hosp_clean["kelas"].value_counts()

kelas
B    19
C     5
A     3
Name: count, dtype: int64

### 4.2 Penghubung rumah sakit ke stasiun
Masalah: tidak ada kolom penghubung. Perbaikan: dipasangkan lewat nama kecamatan, hasilnya location_id.

In [27]:
def kunci_kec(s):
    return s.str.replace(r"\s*\(.*\)", "", regex=True).str.strip()

sites_clean["kec_key"] = kunci_kec(sites_clean["kecamatan"])
hosp_clean["kec_key"]  = kunci_kec(hosp_clean["kecamatan"])

hosp_clean = hosp_clean.merge(sites_clean[["kec_key", "location_id"]],
                              on="kec_key", how="left", validate="one_to_one")
hosp_clean["location_id"].isna().sum()   # harus 0

np.int64(0)

In [28]:
hosp_clean[["hospital_id", "kecamatan", "location_id"]]

,hospital_id,kecamatan,location_id
0,RS-XXX-01,Gambir,LOC-JKT-01
1,RS-XXX-02,Cilincing,LOC-JKT-02
2,RS-XXX-03,Cengkareng,LOC-JKT-03
3,RS-XXX-04,Jekan Raya,LOC-KTG-01
4,RS-XXX-05,Mentawa Baru Ketapang,LOC-KTG-02
5,RS-XXX-06,Cisarua,LOC-BGR-01
6,RS-XXX-07,Bogor Tengah,LOC-BGR-02
7,RS-XXX-08,Mlati,LOC-YKY-01
8,RS-XXX-09,Indrapuri,LOC-IDP-01
9,RS-XXX-10,Palembayan,LOC-KTB-01


## 5. Simpan hasil

In [29]:
os.makedirs("../data/clean/master", exist_ok=True)
sites_clean.to_csv("../data/clean/master/sites_clean.csv", index=False)
sens_clean.to_csv("../data/clean/master/sensors_clean.csv", index=False)
hosp_clean.to_csv("../data/clean/master/hospitals_clean.csv", index=False)
os.listdir("../data/clean/master")

['hospitals_clean.csv', 'sensors_clean.csv', 'sites_clean.csv']